# Combining DataFrames

Concatenation can be used if the dataframes already have the same format.

In [1]:
import numpy as np
import pandas as pd

In [2]:
data_one={'A':['A0','A1','A2','A3'],'B':['B0','B1','B2','B3']}
data_two={'C':['C0','C1','C2','C3'],'D':['D0','D1','D2','D3']}

In [3]:
one = pd.DataFrame(data_one)
two = pd.DataFrame(data_two)

In [4]:
pd.concat([one,two],axis=1)

,A,B,C,D
0,A0,B0,C0,D0
1,A1,B1,C1,D1
2,A2,B2,C2,D2
3,A3,B3,C3,D3


In [5]:
pd.concat([two,one],axis=1)

,C,D,A,B
0,C0,D0,A0,B0
1,C1,D1,A1,B1
2,C2,D2,A2,B2
3,C3,D3,A3,B3


To join the dataframes by rows, you specify axis=0. But since the columns are not the same in both dataframes, there is going to be a lot of NaN.

In [6]:
pd.concat([one,two], axis=0)

,A,B,C,D
0,A0,B0,NaN,NaN
1,A1,B1,NaN,NaN
2,A2,B2,NaN,NaN
3,A3,B3,NaN,NaN
0,NaN,NaN,C0,D0
1,NaN,NaN,C1,D1
2,NaN,NaN,C2,D2
3,NaN,NaN,C3,D3


The presence of so many NaN, exactly at the overlapping, means that it was probably a bad idea.

If you are indeed sure that the concatenation is correct and you want only two columns in the result, you may rename the columns on one of the dataframes.

In [7]:
two.columns.values

<StringArray>
['C', 'D']
Length: 2, dtype: str

In [8]:
two.columns = one.columns
two

,A,B
0,C0,D0
1,C1,D1
2,C2,D2
3,C3,D3


In [9]:
pd.concat([one,two], axis=0)

,A,B
0,A0,B0
1,A1,B1
2,A2,B2
3,A3,B3
0,C0,D0
1,C1,D1
2,C2,D2
3,C3,D3


To have a reset index, you may change the index, in order to use it to slice or select

In [10]:
new_df = pd.concat([one, two], axis=0)
new_df.index = range(len(new_df))
new_df

,A,B
0,A0,B0
1,A1,B1
2,A2,B2
3,A3,B3
4,C0,D0
5,C1,D1
6,C2,D2
7,C3,D3


## Merge dataframes

### Inner join

In [17]:
registrations = pd.DataFrame({'reg_id':[1,2,3,4], 'name':['Andrew','Bobo','Claire','David']})
logins = pd.DataFrame({'log_id':[1,2,3,4], 'name':['Xavier','Andrew','Yolanda','Bobo']})

In [18]:
registrations

,reg_id,name
0,1,Andrew
1,2,Bobo
2,3,Claire
3,4,David


In [19]:
logins

,log_id,name
0,1,Xavier
1,2,Andrew
2,3,Yolanda
3,4,Bobo


In [20]:
pd.merge(registrations,logins,how='inner',on='name')

,reg_id,name,log_id
0,1,Andrew,2
1,2,Bobo,4


### Right and left join

'right' or 'left' values for the how parameter refer to which one of the two dataframes drives the merge. Whichever it is, its values will be all present in the result, whereas the values from the other dataframe will only be present if they have value for the driving column.

So, for our example, registrations is left and logins is right. On a 'left' merge, all rows from registrations will be in the result; columns from logins will have values for the rows whose "name" is both in logins and registrations, the rest of the rows will have logins columns as NaN.

On a 'right' merge, all rows from logins will be in the result, having registrations values for those 'name' which are in both.

In [22]:
pd.merge(registrations,logins,how='left',on='name')

,reg_id,name,log_id
0,1,Andrew,2.0
1,2,Bobo,4.0
2,3,Claire,NaN
3,4,David,NaN


In [23]:
pd.merge(registrations,logins,how='right',on='name')

,reg_id,name,log_id
0,NaN,Xavier,1
1,1.0,Andrew,2
2,NaN,Yolanda,3
3,2.0,Bobo,4


### Outer join

how='outer' means get everything from both dataframes, setting missing values for whatever is not present in both.

In [24]:
pd.merge(logins,registrations,how='outer',on='name')

,log_id,name,reg_id
0,2.0,Andrew,1.0
1,4.0,Bobo,2.0
2,NaN,Claire,3.0
3,NaN,David,4.0
4,1.0,Xavier,NaN
5,3.0,Yolanda,NaN


### joining on an index instead of a column

In [25]:
registrations = registrations.set_index('name')

In [26]:
# now merge by registration's index and logins 'name' column 
pd.merge(logins,registrations,left_on='name',right_index=True,how='inner')

,log_id,name,reg_id
1,2,Andrew,1
3,4,Bobo,2


In [27]:
registrations=registrations.reset_index()

In [28]:
registrations

,name,reg_id
0,Andrew,1
1,Bobo,2
2,Claire,3
3,David,4


In [29]:
registrations.columns.values

<StringArray>
['name', 'reg_id']
Length: 2, dtype: str

In [30]:
registrations.columns = ['reg_name', 'reg_id']

In [31]:
registrations

,reg_name,reg_id
0,Andrew,1
1,Bobo,2
2,Claire,3
3,David,4


In [33]:
pd.merge(registrations,logins,how='inner', left_on='reg_name',right_on='name')

,reg_name,reg_id,log_id,name
0,Andrew,1,2,Andrew
1,Bobo,2,4,Bobo


In [34]:
pd.merge(registrations,logins,how='inner', left_on='reg_name',right_on='name').drop('reg_name', axis=1)

,reg_id,log_id,name
0,1,2,Andrew
1,2,4,Bobo
